# Reproducing Figure 8 under the paper's own protocol

Figure 8 of the QUANT paper sweeps depth against quantiles per interval, and the text draws two conclusions from it:

> accuracy tends to increase as depth increases, up to a depth of approximately d = 6, and then decreases. The same effect is not evident for m/4 or more quantiles per interval.

> Configurations using more quantiles per interval appear to be relatively immune to this effect.

That pattern matters beyond the figure. It justifies the depth-6 default, it supports the claim that larger intervals are more informative, and it is the basis for the untested suggestion about weighted feature sampling that we tested separately.

Our weighting experiment found no trace of it, but used a different protocol, so it could not distinguish *the pattern is not real* from *the pattern depends on the protocol*. This notebook removes that ambiguity.

### Protocol

Matched to Sect. 4.2 as closely as the paper allows:

| | paper | here |
|---|---|---|
| splits | stratified 5-fold CV, 80/20, on pooled train+test | same |
| eligibility | >= 100 training cases at 80/20, >= 5 per class | same |
| eligible pool | **100** of 112 | **100** of 112 |
| sample | 50 drawn at random | 50 drawn at random, seed 0 |
| grid | depth 4-8 x {m, m/2, m/4, m/8, m/16} | same |
| classifier | 200 trees, `max_features=0.1`, entropy | same |

The eligible pool matching at exactly 100 is a strong check that the filter is being read the same way.

**What cannot be matched:** the paper does not list which 50 datasets it sampled, so ours is a different draw from the same pool.

6,250 fits, 18 hours wall clock.

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import wilcoxon

pd.set_option("display.width", 140)

import os
from pathlib import Path

ROOT = Path(os.environ.get("QUANT_ROOT", Path.cwd().parent))
raw = pd.read_csv(f"{ROOT}/results/fig8_results.csv")
raw = raw[raw["fold"] >= 0].drop_duplicates(["dataset", "depth", "div", "fold"])

DEPTHS = sorted(raw["depth"].unique())
DIVS = sorted(raw["div"].unique())
LABEL = {d: ("m" if d == 1 else f"m/{d}") for d in DIVS}

# per-dataset mean over the 5 folds: the unit of analysis
per = raw.groupby(["dataset", "div", "depth"])["accuracy"].mean()

# datasets too short to reach every requested depth
sat = raw.groupby("dataset")["saturated_from"].first()
FULL = sorted(sat[sat.isna()].index)
SHORT = sorted(sat[sat.notna()].index)

print(f"{raw['dataset'].nunique()} datasets, {raw['fold'].nunique()} folds, "
      f"{len(raw)} fits")
print(f"{len(FULL)} reach every depth; {len(SHORT)} saturate early")
print(f"  saturating: {', '.join(SHORT)}")

50 datasets, 5 folds, 6250 fits
40 reach every depth; 10 saturate early
  saturating: Chinatown, ECG200, MedicalImages, MiddlePhalanxTW, MoteStrain, PhalangesOutlinesCorrect, ProximalPhalanxOutlineAgeGroup, ProximalPhalanxOutlineCorrect, SonyAIBORobotSurface2, SyntheticControl


## 1. The surface

First the raw equivalent of Fig. 8's left panel, and the thing to notice before any test: how much the whole surface varies.

In [2]:
surface = (per.reset_index()
           .pivot_table(index="depth", columns="div", values="accuracy"))
surface.columns = [LABEL[c] for c in surface.columns]

display(surface.round(4))

span = surface.values.max() - surface.values.min()
print(f"range across all {surface.size} configurations: "
      f"{surface.values.min():.4f} to {surface.values.max():.4f} "
      f"(span {span:.4f})")
print(f"paper's Fig. 8 range: roughly 0.9100 to 0.9170 (span 0.0070)")

,m,m/2,m/4,m/8,m/16
depth,,,,,
4,0.9005,0.9012,0.9006,0.8991,0.8965
5,0.9020,0.9014,0.9023,0.9007,0.8991
6,0.9016,0.9024,0.9024,0.9017,0.8993
7,0.9022,0.9021,0.9014,0.9018,0.8998
8,0.9024,0.9017,0.9008,0.9020,0.8994


range across all 25 configurations: 0.8965 to 0.9024 (span 0.0059)
paper's Fig. 8 range: roughly 0.9100 to 0.9170 (span 0.0070)


Two things are already clear.

**The accuracy level is close to the paper's** — about 0.90 here against about 0.914 there, on a different random 50 from the same pool. That is reassuring about the protocol.

**The whole surface is flat.** Every one of the 25 configurations lies within a span comparable to the paper's own. Whatever is happening here, it is small.

## 2. Is there a decline from depth 6, and is it confined to low quantile counts?

This is the actual claim. Test it per line: paired Wilcoxon on each dataset's depth-6 and depth-8 accuracy, within each quantile setting. n = 50, and the five tests are independent of one another in the sense that matters (different feature configurations), though all drawn on the same datasets — so correct for five comparisons.

In [3]:
D_REF, D_LAST = 6, DEPTHS[-1]
ALPHA = 0.05
BONF = ALPHA / len(DIVS)


def decline_test(datasets, d_from=D_REF, d_to=D_LAST):
    rows = []
    for div in DIVS:
        a = per.loc[(datasets, div, d_to)].droplevel([1, 2])
        b = per.loc[(datasets, div, d_from)].droplevel([1, 2])
        common = a.index.intersection(b.index)
        a, b = a[common], b[common]
        diff = a - b
        rows.append({
            "quantiles": LABEL[div],
            "n": len(diff),
            f"depth {d_from}": b.mean(),
            f"depth {d_to}": a.mean(),
            "change": diff.mean(),
            "declining": int((diff < 0).sum()),
            "p": wilcoxon(a, b).pvalue,
            "paper says": {1: "immune", 2: "immune", 4: "immune",
                           8: "declines", 16: "declines"}[div],
        })
    return pd.DataFrame(rows).set_index("quantiles")


print(f"depth {D_REF} -> {D_LAST}, all {raw['dataset'].nunique()} datasets")
print(f"Bonferroni threshold for {len(DIVS)} comparisons: {BONF:.4f}\n")
display(decline_test(sorted(raw['dataset'].unique())).round(4))

depth 6 -> 8, all 50 datasets
Bonferroni threshold for 5 comparisons: 0.0100



,n,depth 6,depth 8,change,declining,p,paper says
quantiles,,,,,,,
m,50,0.9016,0.9024,0.0008,20,0.8741,immune
m/2,50,0.9024,0.9017,-0.0008,24,0.3356,immune
m/4,50,0.9024,0.9008,-0.0016,27,0.0402,immune
m/8,50,0.9017,0.9020,0.0003,18,0.7662,declines
m/16,50,0.8993,0.8994,0.0001,19,0.9940,declines


## 3. Does the depth cap explain anything?

`quant.py` caps depth internally at `min(depth, int(log2(L)) + 1)` for each representation of length `L`. On short series the deeper levels therefore produce an **identical** transform, which would flatten any decline by construction.

Repeat the test on only those datasets long enough to use every depth requested.

In [4]:
print(f"depth {D_REF} -> {D_LAST}, the {len(FULL)} datasets that reach every depth\n")
display(decline_test(FULL).round(4))

depth 6 -> 8, the 40 datasets that reach every depth



,n,depth 6,depth 8,change,declining,p,paper says
quantiles,,,,,,,
m,40,0.9040,0.9047,0.0008,17,0.8542,immune
m/2,40,0.9052,0.9035,-0.0017,21,0.0778,immune
m/4,40,0.9043,0.9027,-0.0016,22,0.0350,immune
m/8,40,0.9035,0.9038,0.0003,14,0.8290,declines
m/16,40,0.9012,0.9010,-0.0002,17,0.7390,declines


## 4. Does the rise to depth 6 happen?

The other half of the paper's description. If accuracy climbs from depth 4 to 6 that at least supports the choice of 6 as the default, even if nothing falls afterwards.

In [5]:
print(f"depth {DEPTHS[0]} -> {D_REF}, the {len(FULL)} datasets that reach every depth\n")
display(decline_test(FULL, d_from=DEPTHS[0], d_to=D_REF).round(4))

depth 4 -> 6, the 40 datasets that reach every depth



,n,depth 4,depth 6,change,declining,p,paper says
quantiles,,,,,,,
m,40,0.9027,0.9040,0.0012,10,0.2536,immune
m/2,40,0.9035,0.9052,0.0016,12,0.2781,immune
m/4,40,0.9025,0.9043,0.0018,11,0.0684,immune
m/8,40,0.9013,0.9035,0.0023,10,0.0919,declines
m/16,40,0.8987,0.9012,0.0025,11,0.1687,declines


## 5. Every depth transition, for completeness

Rather than only the endpoints, test each consecutive step. If the curve really peaks at 6, the steps before it should be positive and the steps after negative.

In [6]:
rows = []
for div in DIVS:
    for d_from, d_to in zip(DEPTHS, DEPTHS[1:]):
        a = per.loc[(FULL, div, d_to)].droplevel([1, 2])
        b = per.loc[(FULL, div, d_from)].droplevel([1, 2])
        common = a.index.intersection(b.index)
        a, b = a[common], b[common]
        rows.append({
            "quantiles": LABEL[div],
            "step": f"{d_from}->{d_to}",
            "change": (a - b).mean(),
            "p": wilcoxon(a, b).pvalue,
        })

steps = pd.DataFrame(rows)
n_tests = len(steps)

print("mean change at each step")
display(steps.pivot(index="step", columns="quantiles",
                    values="change").reindex(
    columns=[LABEL[d] for d in DIVS]).round(4))

print("p")
display(steps.pivot(index="step", columns="quantiles",
                    values="p").reindex(
    columns=[LABEL[d] for d in DIVS]).round(3))

print(f"{n_tests} tests; Bonferroni threshold {ALPHA / n_tests:.5f}")
sig = steps[steps["p"] < ALPHA / n_tests]
print(f"surviving correction: {len(sig)}")
if len(sig):
    display(sig.round(4))

mean change at each step


quantiles,m,m/2,m/4,m/8,m/16
step,,,,,
4->5,0.0019,0.0006,0.0015,0.0012,0.0025
5->6,-0.0006,0.0010,0.0003,0.0011,-0.0001
6->7,0.0005,-0.0012,-0.0008,-0.0000,0.0003
7->8,0.0003,-0.0005,-0.0008,0.0003,-0.0005


p


quantiles,m,m/2,m/4,m/8,m/16
step,,,,,
4->5,0.089,0.274,0.142,0.158,0.081
5->6,0.784,0.308,0.734,0.666,0.493
6->7,0.922,0.362,0.943,0.754,0.855
7->8,0.784,0.451,0.308,0.389,0.581


20 tests; Bonferroni threshold 0.00250
surviving correction: 0


## 6. How large would the paper's effect be?

Rather than only asking whether we see their effect, ask whether this experiment *could* have seen it. Read off Fig. 8: the m/16 line runs from roughly 0.9100 at depth 4 to about 0.9125 at depth 6, then down to about 0.9110 at depth 8 — a post-peak decline of roughly 0.0015.

Compare that against the spread of per-dataset differences here.

In [7]:
PAPER_DECLINE = 0.0015   # approximate, read from Fig. 8's m/16 line

rows = []
for div in DIVS:
    a = per.loc[(FULL, div, D_LAST)].droplevel([1, 2])
    b = per.loc[(FULL, div, D_REF)].droplevel([1, 2])
    diff = (a - b).dropna()
    sem = diff.std(ddof=1) / np.sqrt(len(diff))
    rows.append({
        "quantiles": LABEL[div],
        "observed change": diff.mean(),
        "std of per-dataset change": diff.std(ddof=1),
        "standard error": sem,
        "95% CI lower": diff.mean() - 1.96 * sem,
        "95% CI upper": diff.mean() + 1.96 * sem,
    })

ci = pd.DataFrame(rows).set_index("quantiles")
display(ci.round(4))

print(f"the paper's apparent post-peak decline is about {-PAPER_DECLINE:+.4f}")
print("\nis that value inside our 95% confidence interval for each line?")
for q, r in ci.iterrows():
    inside = r["95% CI lower"] <= -PAPER_DECLINE <= r["95% CI upper"]
    print(f"  {q:<6} [{r['95% CI lower']:+.4f}, {r['95% CI upper']:+.4f}]  "
          f"{'yes - we cannot rule it out' if inside else 'no - excluded'}")

,observed change,std of per-dataset change,standard error,95% CI lower,95% CI upper
quantiles,,,,,
m,0.0008,0.0078,0.0012,-0.0017,0.0032
m/2,-0.0017,0.0074,0.0012,-0.0040,0.0006
m/4,-0.0016,0.0070,0.0011,-0.0038,0.0005
m/8,0.0003,0.0086,0.0014,-0.0024,0.0029
m/16,-0.0002,0.0079,0.0012,-0.0027,0.0022


the paper's apparent post-peak decline is about -0.0015

is that value inside our 95% confidence interval for each line?
  m      [-0.0017, +0.0032]  yes - we cannot rule it out
  m/2    [-0.0040, +0.0006]  yes - we cannot rule it out
  m/4    [-0.0038, +0.0005]  yes - we cannot rule it out
  m/8    [-0.0024, +0.0029]  yes - we cannot rule it out
  m/16   [-0.0027, +0.0022]  yes - we cannot rule it out


This is the honest crux. If the paper's effect size sits inside our confidence intervals, then our experiment is consistent both with their pattern and with no pattern at all, and the correct conclusion is that neither study has the resolution to settle it.

## 7. Cost

Fig. 8's right panel. This part of the figure is about compute, where the differences are large and unambiguous.

In [8]:
cost = (raw.pivot_table(index="depth", columns="div",
                        values="seconds", aggfunc="sum") / 60)
cost.columns = [LABEL[c] for c in cost.columns]

print("total compute, minutes (multi-core; the paper's figures are single-core)")
display(cost.round(1))

print("relative to the cheapest configuration")
display((cost / cost.values.min()).round(1))

feat = raw.pivot_table(index="depth", columns="div",
                       values="n_features", aggfunc="mean")
feat.columns = [LABEL[c] for c in feat.columns]
print("mean feature count")
display(feat.round(0))

total compute, minutes (multi-core; the paper's figures are single-core)


,m,m/2,m/4,m/8,m/16
depth,,,,,
4,58.6,27.2,14.7,9.4,7.4
5,83.8,37.8,20.3,12.7,9.5
6,109.5,50.6,27.1,17.3,12.6
7,124.2,64.7,35.7,23.0,16.7
8,147.6,78.0,45.1,30.5,22.9


relative to the cheapest configuration


,m,m/2,m/4,m/8,m/16
depth,,,,,
4,7.9,3.7,2.0,1.3,1.0
5,11.4,5.1,2.8,1.7,1.3
6,14.8,6.9,3.7,2.3,1.7
7,16.8,8.8,4.8,3.1,2.3
8,20.0,10.6,6.1,4.1,3.1


mean feature count


,m,m/2,m/4,m/8,m/16
depth,,,,,
4,11546.0,5796.0,2922.0,1487.0,769.0
5,15197.0,7653.0,3883.0,1995.0,1055.0
6,18899.0,9558.0,4893.0,2568.0,1417.0
7,22567.0,11491.0,5974.0,3237.0,1909.0
8,26080.0,13395.0,7114.0,4031.0,2590.0


## 8. Summary

**What matched.** The eligibility filter returns exactly the 100 datasets the paper reports, and the accuracy level is close to theirs on a different random 50 from the same pool. The protocol is faithfully reproduced.

**What did not.** The depth-by-quantile interaction is not where the paper puts it. The two settings it says decline, m/8 and m/16, move by +0.0003 and +0.0001 from depth 6 to 8. The only line with p < 0.05 is m/4, which the paper calls immune — and that single result is one of five tests and does not survive correction.

**The crucial qualification.** The entire 25-configuration surface spans about 0.006, and the paper's own spans about 0.007. An effect of that size is at the edge of what either study can resolve. Section 6 asks directly whether the paper's apparent decline lies inside our confidence intervals, and it does for every one of the five lines. The honest statement is therefore that **this experiment cannot distinguish their pattern from no pattern**, not that their pattern is wrong.

**Why it matters either way.** Fig. 8 carries weight in the paper: it justifies the depth-6 default, supports the claim that larger intervals are more informative, and motivates the weighted-sampling suggestion. A figure drawn without error bars, whose features are the same size as the surface's total variation, is weak support for any of those.

### Caveats

- **A different 50 datasets.** The paper does not say which it sampled. Ours is seed 0 on the same eligible pool. Two random draws of 50 from 100 datasets can differ noticeably when effects are this small.
- **Ten of our 50 saturate before depth 8** and were analysed separately (sections 3 onward use the 40 that do not).
- **Multi-core timings**, not comparable with the paper's single-core figures.
- **The paper's effect size was read off a figure**, not taken from a table, so the comparison in section 6 is approximate.
- Five-fold CV on pooled data gives one accuracy per dataset per configuration; per-dataset variance across folds is not propagated into the tests.